In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt

In [11]:
data = pd.read_excel("Data/dataset_comp_1222_without_ceg_ogn.xlsx")

# 1) Calculate historical returns 

In [ ]:
# # Define the tickers and date range
# tickers = list(data['SYMBOL'].values) # Add more tickers as needed
# start_date = "2015-11-01"
# end_date = "2021-01-01"

# # Download the data
# yahoo_data = yf.download(tickers, start=start_date, end=end_date, group_by='ticker', auto_adjust=False, progress=False)

# # Extract adjusted close prices
# adj_close_bt = pd.DataFrame()

# # If only one ticker, yfinance doesn't use a multi-indexed DataFrame, handle accordingly
# if len(tickers) == 1:
#     adj_close_bt[tickers[0]] = yahoo_data['Adj Close']
# else:
#     for ticker in tickers:
#         try:
#             adj_close_bt[ticker] = yahoo_data[ticker]['Adj Close']
#         except KeyError:
#             print(f"No data found for {ticker}.")

# # Display the result
# print(adj_close_bt.head())

12 Failed downloads:
['SBNY', 'CEG', 'OGN']: YFPricesMissingError('possibly delisted; no price data found  (1d 2015-11-01 -> 2021-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1446350400, endDate = 1609477200")')
['MRO', 'CTLT', 'PXD', 'WRK', 'DISH', 'ATVI', 'SIVBQ']: YFTzMissingError('possibly delisted; no timezone found')
['DFS']: YFPricesMissingError('possibly delisted; no price data found  (1d 2015-11-01 -> 2021-01-01) (Yahoo error = "No data found, symbol may be delisted")')
['BF.B']: YFPricesMissingError('possibly delisted; no price data found  (1d 2015-11-01 -> 2021-01-01)')

In [ ]:
# #print(data.loc[data['SYMBOL'] == 'BF.B'])
# # Define the tickers and date range
# tickers = ['BF-B'] # Add more tickers as needed
# start_date = "2015-11-01"
# end_date = "2021-01-01"

# # Download the data
# data_bfb = yf.download(tickers, start=start_date, end=end_date, group_by='ticker', auto_adjust=False, progress=False)

# # Extract adjusted close prices
# adj_close_bfb = pd.DataFrame()

# # If only one ticker, yfinance doesn't use a multi-indexed DataFrame, handle accordingly
# if len(tickers) == 1:
#     adj_close_bfb[tickers[0]] = data_bfb['BF-B']['Adj Close']
# else:
#     for ticker in tickers:
#         try:
#             adj_close_bfb[ticker] = data_bfb[ticker]['Adj Close']
#         except KeyError:
#             print(f"No data found for {ticker}.")

# # Display the result
# #print(adj_close_bfb.head())
# adj_close_bt['BF.B'] = adj_close_bfb.values

In [4]:
# adj_close_bt.to_excel("Data/adj_price_yahoo_comp_1222_historical.xlsx")

In [5]:
adj_close_bt = pd.read_excel("Data/adj_price_yahoo_comp_1222_historical.xlsx")
adj_close_bt.index = adj_close_bt.Date
adj_close_bt.drop(columns='Date', inplace=True)


'CEG', 'OGN'

['DFS']: YFPricesMissingError('possibly delisted; no price data found  (1d 2015-11-01 -> 2021-01-01) (Yahoo error = "No data found, symbol may be delisted")')

     

In [ ]:
price_historical = pd.read_excel('Data/price_div_comp_1222_historical.xlsm', sheet_name='CLOSE PRICE', header = 4)

price_historical.columns = price_historical.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip()

price_historical = price_historical.iloc[217:1566]

div_rate = pd.read_excel('Data/price_div_comp_1222_historical.xlsm', sheet_name='DIV RATE', header = 4)
div_rate.columns = div_rate.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip()
div_rate = div_rate.iloc[217:1566]

div_date_historical = pd.read_excel('Data/price_div_comp_1222_historical.xlsm', sheet_name='DIV DATE', header = 4)
div_date_historical.columns = div_date_historical.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip()
div_date_historical = div_date_historical.iloc[217:1566]

div_date_historical.columns = price_historical.columns
div_rate.columns = price_historical.columns

price_historical.index = price_historical.Code
div_rate.index = div_rate.Code
div_date_historical.index = div_date_historical.Code

price_historical = price_historical.iloc[:, 1:]
div_rate = div_rate.iloc[:, 1:]
div_date_historical = div_date_historical.iloc[:, 1:]

# --- Step 1: Calculate adjustment factors
adj_factors = pd.DataFrame(1.0, index=price_historical.index, columns=price_historical.columns)

for company in price_historical.columns:
    for i in range(1, len(price_historical)):
        date = price_historical.index[i]
        prev_date = price_historical.index[i - 1]

        # If ex-dividend happens on this day
        if pd.notna(div_date_historical.at[date, company]):
            div = div_rate.at[date, company]
            price_prev = price_historical.at[prev_date, company]
            if price_prev and price_prev != 0:
                factor = (price_prev - div) / price_prev
                adj_factors.at[date, company] = factor

# --- Step 2: Calculate cumulative adjustment factors in reverse (like Yahoo)
cum_factors = adj_factors.iloc[::-1].cumprod().iloc[::-1]

# --- Step 3: Build adjusted prices
adjusted_prices_calculated = price_historical * cum_factors

#print(data.loc[data['SYMBOL'] == 'WRK', 'NAME'])
#print(div_rate['96699P'].unique())
#print(price_historical['96699P'])
#print(adjusted_prices_calculated['96699P'])
adj_close_bt['WRK'] = adjusted_prices_calculated.loc[adj_close_bt.index, '96699P'].values

#print(data.loc[data['SYMBOL'] == 'CTLT', 'NAME'])
#print(div_rate['8866F3'].unique())
#print(price_historical['8866F3'])
#print(adjusted_prices_calculated['8866F3'])
adj_close_bt['CTLT'] = adjusted_prices_calculated.loc[adj_close_bt.index, ['8866F3']].values

#print(data.loc[data['SYMBOL'] == 'MRO', 'NAME'])
#print(div_rate['544682'].unique())
#print(price_historical['544682'])
#print(adjusted_prices_calculated['544682'])
adj_close_bt['MRO'] = adjusted_prices_calculated.loc[adj_close_bt.index, '544682'].values

#print(data.loc[data['SYMBOL'] == 'PXD', 'NAME'])
#print(div_rate['895705'].unique())
#print(price_historical['895705'])
adj_close_bt['PXD'] = adjusted_prices_calculated.loc[adj_close_bt.index, '895705'].values

#print(data.loc[data['SYMBOL'] == 'SBNY', ['NAME','TYPE']])
#print(div_rate['28709C'].unique())
#print(price_historical['28709C'])
adj_close_bt['SBNY'] = adjusted_prices_calculated.loc[adj_close_bt.index, '28709C'].values

#print(data.loc[data['SYMBOL'] == 'ATVI', ['NAME','TYPE']])
#print(div_rate['312367'].unique())
#print(price_historical['312367'])
adj_close_bt['ATVI'] = adjusted_prices_calculated.loc[adj_close_bt.index, '312367'].values

#print(data.loc[data['SYMBOL'] == 'DISH', ['NAME','TYPE']])
#print(div_rate['135448'].unique())
#print(price_historical['135448'])
adj_close_bt['DISH'] = adjusted_prices_calculated.loc[adj_close_bt.index, '135448'].values

#print(data.loc[data['SYMBOL'] == 'SIVBQ', ['NAME','TYPE']])
#print(div_rate['518628'].unique())
#print(price_historical['518628'])
adj_close_bt['SIVBQ'] = adjusted_prices_calculated.loc[adj_close_bt.index, '518628'].values

print(div_rate['50642F'].unique())
print(price_historical['50642F'])
adj_close_bt['DFS'] = adjusted_prices_calculated.loc[adj_close_bt.index, '50642F'].values

[ nan 0.28 0.3  0.35 0.4  0.44]
Code
2015-11-02    56.28
2015-11-03    56.35
2015-11-04    56.37
2015-11-05    56.91
2015-11-06    57.61
              ...  
2020-12-25    88.29
2020-12-28    88.29
2020-12-29    88.02
2020-12-30    89.30
2020-12-31    90.53
Name: 50642F, Length: 1349, dtype: float64


In [ ]:
# ceg and ogn already excluded

In [ ]:
#right now my log returns go from 15 dec 2020 to 15 dec 2021 (so the data goes back from 15 nov 2020)
# I could back test three years before so :

# 15 nov 2017 to 15 nov 2018
# 15 nov 2018 to 15 nov 2019
# 15 nov 2019 to 15 nov 2020

In [9]:
from datetime import datetime
adj_close_bt_above_2018 = adj_close_bt.loc[adj_close_bt.index > datetime(2017, 11, 15)]
cols_with_nans = adj_close_bt_above_2018.columns[adj_close_bt_above_2018.isna().any()]
print("Columns with NaNs:", cols_with_nans.tolist())

nan_columns = adj_close_bt_above_2018.loc[:, adj_close_bt_above_2018.isna().any(axis=0)]

for col in nan_columns.columns:
    first_valid = nan_columns[col].first_valid_index()
    print(f"First non-NaN in column '{col}': {first_valid}") # these stocks had later IPO

Columns with NaNs: ['DAY', 'MRNA', 'DOW', 'FOXA', 'CARR', 'OTIS', 'CTVA', 'OGN', 'CEG', 'VICI']
First non-NaN in column 'DAY': 2018-04-26 00:00:00
First non-NaN in column 'MRNA': 2018-12-07 00:00:00
First non-NaN in column 'DOW': 2019-03-20 00:00:00
First non-NaN in column 'FOXA': 2019-03-12 00:00:00
First non-NaN in column 'CARR': 2020-03-19 00:00:00
First non-NaN in column 'OTIS': 2020-03-19 00:00:00
First non-NaN in column 'CTVA': 2019-05-24 00:00:00
First non-NaN in column 'OGN': None
First non-NaN in column 'CEG': None
First non-NaN in column 'VICI': 2018-01-02 00:00:00


VICI is okay, OGN and CEG I have to remove them in any case. For the rest I can take the optimized portfolio, and exclude these 7 stocks and then I can see how does it change in practice the optimised portfolio. After doing this I can evaluate the portfolio back in time. 

In [12]:

nan_backtest = ['DAY', 'MRNA', 'DOW', 'FOXA', 'CARR', 'OTIS', 'CTVA', 'VICI']
for stock in nan_backtest:
    stock_type_miss = data.loc[data['SYMBOL'] == stock, 'TYPE'].values
    if price_historical[stock_type_miss].first_valid_index() < nan_columns[stock].first_valid_index():
        print(stock, price_historical[stock_type_miss].first_valid_index())

VICI 2017-10-17 00:00:00


In [20]:
vici = price_historical['9174A0']
vici = vici[vici.index >= datetime(2017, 12, 1)]

In [22]:
adj_close_bt = adj_close_bt[(adj_close_bt.index >= datetime(2017, 12, 1))]

In [25]:
adj_close_bt['VICI'] = vici

/var/folders/n9/cy4s67js3v58hhczg4lxj5jr0000gq/T/ipykernel_91944/1411507496.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  adj_close_bt['VICI'] = vici


In [29]:
adj_close_bt = adj_close_bt.drop(columns=['OGN', 'CEG', 'DAY', 'MRNA', 'DOW', 'FOXA', 'CARR', 'OTIS', 'CTVA'])